In [84]:
from cirq_sic import *

In [126]:
d = 3
D = wh_operators(d)["D"].reshape(d**2,d,d)

In [127]:
def char_probabilities(rho, D):
    return (abs(np.array([(O@rho).trace() for O in D]))**2/(d*(rho@rho).trace())).real

def povm_probabilities(rho, D):
    d = rho.shape[0]
    return np.array([(O @ rho @ O.conj().T @ rho).trace() for O in D]).real/d

def renyi_entropy(p, alpha=2):
    return (1/(1-alpha))*np.log(sum(p**alpha))

def renyi_se(rho, D, alpha=2):
    d = rho.shape[0]
    p = char_probabilities(rho, D)
    return (renyi_entropy(p, alpha) - np.log(d) - np.log((rho@rho).trace())).real

def renyi_se_max(d, alpha):
    return (1/(1-alpha))*np.log((1+(d-1)*(d+1)**(1-alpha))/d)

In [132]:
rho = rand_dm(d, r=2)
p = char_probabilities(rho, D);
p, sum(p)

(array([0.564, 0.032, 0.032, 0.001, 0.089, 0.096, 0.001, 0.096, 0.089]),
 np.float64(0.9999999999999998))

In [133]:
renyi_se(rho, D, 2)

np.float64(0.46553421668565265)

In [134]:
-np.log(sum([abs((O@rho).trace())**4 for O in D])/(d*(rho@rho).trace())).real

np.float64(0.46553421668565215)

In [135]:
phi = load_sic_fiducial(d)
rho = np.outer(phi, phi.conj())
renyi_se(rho, D, 2), renyi_se_max(d, 2)

(np.float64(0.6931471769500107), np.float64(0.6931471805599453))

In [136]:
renyi_se(np.eye(d)/d, D, 2)

np.float64(0.0)

In [137]:
def depolarize(rho, p):
    d = rho.shape[0]
    return (1-p)*rho + p*np.eye(d)/d

In [138]:
np.array([renyi_se(depolarize(rho, x), D, 2) for x in np.linspace(0, 1, 10)])

array([0.693, 0.676, 0.625, 0.542, 0.434, 0.314, 0.195, 0.093, 0.024,
       0.   ])

In [139]:
rho = rand_dm(d, r=1)
p = char_probabilities(rho, D);
p_ = povm_probabilities(rho, D)
p, p_

(array([0.333, 0.075, 0.075, 0.063, 0.042, 0.154, 0.063, 0.154, 0.042]),
 array([0.333, 0.075, 0.075, 0.063, 0.042, 0.154, 0.063, 0.154, 0.042]))

In [176]:
rho = rand_dm(d, r=2)
p = char_probabilities(rho, D);
p_ = povm_probabilities(rho, D)
p, p_

(array([0.424, 0.072, 0.072, 0.06 , 0.123, 0.033, 0.06 , 0.033, 0.123]),
 array([0.262, 0.092, 0.092, 0.083, 0.132, 0.062, 0.083, 0.062, 0.132]))

In [177]:
w = np.exp(2*np.pi*1j/d)
def symplectic_form(a, b):
    return a[1]*b[0] - a[0]*b[1]

F = np.array([[w**symplectic_form(a,b) for b in np.ndindex(d,d)] for a in np.ndindex(d, d)])
F

array([[ 1. +0.j   ,  1. +0.j   ,  1. +0.j   ,  1. +0.j   ,  1. +0.j   ,
         1. +0.j   ,  1. +0.j   ,  1. +0.j   ,  1. +0.j   ],
       [ 1. +0.j   ,  1. +0.j   ,  1. +0.j   , -0.5+0.866j, -0.5+0.866j,
        -0.5+0.866j, -0.5-0.866j, -0.5-0.866j, -0.5-0.866j],
       [ 1. +0.j   ,  1. +0.j   ,  1. +0.j   , -0.5-0.866j, -0.5-0.866j,
        -0.5-0.866j, -0.5+0.866j, -0.5+0.866j, -0.5+0.866j],
       [ 1. +0.j   , -0.5-0.866j, -0.5+0.866j,  1. +0.j   , -0.5-0.866j,
        -0.5+0.866j,  1. +0.j   , -0.5-0.866j, -0.5+0.866j],
       [ 1. +0.j   , -0.5-0.866j, -0.5+0.866j, -0.5+0.866j,  1. +0.j   ,
        -0.5-0.866j, -0.5-0.866j, -0.5+0.866j,  1. +0.j   ],
       [ 1. +0.j   , -0.5-0.866j, -0.5+0.866j, -0.5-0.866j, -0.5+0.866j,
         1. +0.j   , -0.5+0.866j,  1. -0.j   , -0.5-0.866j],
       [ 1. +0.j   , -0.5+0.866j, -0.5-0.866j,  1. +0.j   , -0.5+0.866j,
        -0.5-0.866j,  1. +0.j   , -0.5+0.866j, -0.5-0.866j],
       [ 1. +0.j   , -0.5+0.866j, -0.5-0.866j, -0.5+0.866j, -0

In [178]:
(((rho@rho).trace()/d)*(F @ p)).real, p_

(array([0.262, 0.092, 0.092, 0.083, 0.132, 0.062, 0.083, 0.062, 0.132]),
 array([0.262, 0.092, 0.092, 0.083, 0.132, 0.062, 0.083, 0.062, 0.132]))

In [179]:
((1/(d*(rho@rho).trace()))*(F.conj() @ p_)).real, p

(array([0.424, 0.072, 0.072, 0.06 , 0.123, 0.033, 0.06 , 0.033, 0.123]),
 array([0.424, 0.072, 0.072, 0.06 , 0.123, 0.033, 0.06 , 0.033, 0.123]))

In [180]:
renyi_se(rho, D)

np.float64(0.6123591557381216)

In [181]:
-np.log((1/p_[0])*sum(p_**2)), -np.log((1/p[0])*sum(p**2))

(np.float64(0.6123591557381219), np.float64(0.6123591557381219))

In [190]:
alpha = 2
rho = rand_dm(d, r=2)
p = char_probabilities(rho, D);
p_ = povm_probabilities(rho, D)
(renyi_entropy(p, alpha) - np.log(d) - np.log((rho@rho).trace())).real,\
(renyi_entropy(p_, alpha) - np.log(d) - np.log((rho@rho).trace())).real


(np.float64(0.4516582499044871), np.float64(1.543024223283274))

In [196]:
(rho@rho).trace()*(1 - d*(rho@rho).trace()*sum(p**2))

np.complex128(0.21058704733400227+0j)

In [197]:
(1/(d*p[0]))*(1 - (1/p[0])*sum(p**2))

np.float64(0.21058704733400221)

In [198]:
d*(p_[0] - sum(p_**2))

np.float64(0.21058704733400202)